# Module 1 — Drug Corpus (clean version)

**Goal:** build `data/drugs/indian_drugs.json` — a unified corpus of Indian medicines + medical info.

**How:** combine two open datasets
- `darkknight25/medical_medicine_dataset` (Hugging Face) — 700 drugs with uses, side effects, descriptions
- `junioralive/Indian-Medicine-Dataset` (GitHub) — Indian brand names mapped to generic ingredients

**Time:** ~5 min.

**Important:** run cells in order. Don't skip.

## Cell 1 — Bootstrap

In [ ]:
import os
from dataclasses import dataclass
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

@dataclass(frozen=True)
class Paths:
    project_root: Path = Path('/content/drive/MyDrive/prescriptai')
    @property
    def drugs_dir(self): return self.project_root / 'data' / 'drugs'
    @property
    def drugs_json(self): return self.drugs_dir / 'indian_drugs.json'
    @property
    def hf_cache(self): return self.project_root / 'hf_cache'

PATHS = Paths()
os.environ['HF_HOME'] = str(PATHS.hf_cache)
os.environ['TRANSFORMERS_CACHE'] = str(PATHS.hf_cache)
os.environ['HF_HUB_CACHE'] = str(PATHS.hf_cache)
PATHS.drugs_dir.mkdir(parents=True, exist_ok=True)
PATHS.hf_cache.mkdir(parents=True, exist_ok=True)
print('Bootstrap done.')

Mounted at /content/drive
Bootstrap done.


## Cell 2 — Install

In [ ]:
!pip install -q huggingface_hub pandas

## Cell 3 — Download + load Dataset A (medical info)

We download the JSONL file directly (the `datasets` library chokes on a malformed row) and parse it line-by-line, skipping any broken rows. ~699 of 700 will load.

In [ ]:
import json
from huggingface_hub import hf_hub_download

jsonl_path = hf_hub_download(
    repo_id='darkknight25/medical_medicine_dataset',
    filename='medical_medicine_dataset.jsonl',
    repo_type='dataset',
)
print(f'Downloaded to: {jsonl_path}')

ds_a_rows = []
broken = 0
with open(jsonl_path, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            ds_a_rows.append(json.loads(line))
        except json.JSONDecodeError:
            broken += 1

print(f'\nLoaded {len(ds_a_rows)} valid rows ({broken} skipped).')
print(f'Columns: {list(ds_a_rows[0].keys())}')
print(f'\nFirst entry:')
for k, v in ds_a_rows[0].items():
    print(f'  {k}: {str(v)[:120]}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Downloaded to: /content/drive/MyDrive/prescriptai/hf_cache/datasets--darkknight25--medical_medicine_dataset/snapshots/f19d163f93d41aec0273c852140843604fa32f2c/medical_medicine_dataset.jsonl

Loaded 699 valid rows (1 skipped).
Columns: ['id', 'medicine_name', 'description', 'uses', 'side_effects', 'disclaimer']

First entry:
  id: 1
  medicine_name: Paracetamol
  description: Paracetamol is a widely used medicine for pain relief and reducing fever.
  uses: ['Headache', 'Muscle aches', 'Fever']
  side_effects: ['Nausea', 'Liver damage if overdosed']
  disclaimer: This is for general awareness only. Always consult a doctor for medical advice.


## Cell 4 — Download Dataset B (Indian brands)

In [ ]:
import requests
import pandas as pd
from io import StringIO

INDIAN_CSV_URL = 'https://raw.githubusercontent.com/junioralive/Indian-Medicine-Dataset/main/DATA/indian_medicine_data.csv'
r = requests.get(INDIAN_CSV_URL, timeout=30)
r.raise_for_status()

df_b = pd.read_csv(StringIO(r.text))

print(f'Rows: {len(df_b):,}')
print(f'Columns: {df_b.columns.tolist()}')
print(f'\nFirst row:')
print(df_b.iloc[0].to_dict())

Rows: 253,973
Columns: ['id', 'name', 'price(₹)', 'Is_discontinued', 'manufacturer_name', 'type', 'pack_size_label', 'short_composition1', 'short_composition2']

First row:
{'id': 1, 'name': 'Augmentin 625 Duo Tablet', 'price(₹)': 223.42, 'Is_discontinued': False, 'manufacturer_name': 'Glaxo SmithKline Pharmaceuticals Ltd', 'type': 'allopathy', 'pack_size_label': 'strip of 10 tablets', 'short_composition1': 'Amoxycillin  (500mg) ', 'short_composition2': '  Clavulanic Acid (125mg)'}


## Cell 5 — Helper: normalize drug names

`'Paracetamol (500mg)'` and `'paracetamol'` should both map to the same lookup key. We strip dosage info and lowercase.

In [ ]:
import re

def normalize_name(s):
    s = re.sub(r'\([^)]*\)', '', str(s))
    s = re.sub(r'\d+\s*mg', '', s, flags=re.IGNORECASE)
    s = re.sub(r'[^a-zA-Z\s]', ' ', s)
    s = re.sub(r'\s+', ' ', s)
    return s.strip().lower()

# Smoke tests
for test in ['Paracetamol (500mg)', 'Amoxycillin (500mg)', 'Pantoprazole', 'Vitamin D3 60000 IU']:
    print(f'{test!r:35s} -> {normalize_name(test)!r}')

'Paracetamol (500mg)'               -> 'paracetamol'
'Amoxycillin (500mg)'               -> 'amoxycillin'
'Pantoprazole'                      -> 'pantoprazole'
'Vitamin D3 60000 IU'               -> 'vitamin d iu'


## Cell 6 — Build the generic lookup

Every drug in Dataset A becomes an entry keyed by its normalized name. This is what we'll match Indian brands against.

In [ ]:
def stringify(v):
    if isinstance(v, list):
        return ', '.join(str(x) for x in v)
    return str(v) if v else ''

generic_lookup = {}
for row in ds_a_rows:
    name = row.get('medicine_name', '')
    if not name:
        continue
    key = normalize_name(name)
    if not key:
        continue
    generic_lookup[key] = {
        'generic_name': name,
        'description': stringify(row.get('description', '')),
        'uses': stringify(row.get('uses', '')),
        'side_effects': stringify(row.get('side_effects', '')),
    }

print(f'{len(generic_lookup)} generics indexed.')
print(f'Sample keys: {list(generic_lookup.keys())[:10]}')

381 generics indexed.
Sample keys: ['paracetamol', 'ibuprofen', 'amoxicillin', 'cetirizine', 'omeprazole', 'metformin', 'loratadine', 'atenolol', 'clarithromycin', 'domperidone']


## Cell 7 — Match Indian brands to generics

In [ ]:
df_b['_generic'] = df_b['short_composition1'].apply(normalize_name)

available = df_b[df_b['Is_discontinued'].astype(str).str.upper().isin(['FALSE', 'NO', '0'])].copy()
available['_matched'] = available['_generic'].isin(generic_lookup.keys())
matched = available[available['_matched']]

print(f'Total brands in dataset:    {len(df_b):,}')
print(f'Available (not discontinued): {len(available):,}')
print(f'Matched to a generic:        {len(matched):,}')
print(f'Unique generics matched:    {matched["_generic"].nunique()}')

Total brands in dataset:    253,973
Available (not discontinued): 246,068
Matched to a generic:        108,609
Unique generics matched:    255


## Cell 8 — Sample top generics + a few brands each

We don't need 100k entries. ~200 most-common generics × 3 brands each = ~600 brand entries, plus 200 generic-only entries = ~800 total. Plenty for RAG.

In [ ]:
TOP_N_GENERICS = 200
BRANDS_PER_GENERIC = 3

top_generics = matched['_generic'].value_counts().head(TOP_N_GENERICS).index.tolist()

sampled = (
    matched[matched['_generic'].isin(top_generics)]
    .groupby('_generic', group_keys=False)
    .head(BRANDS_PER_GENERIC)
    .reset_index(drop=True)
)

print(f'Top 10 generics: {top_generics[:10]}')
print(f'\nSampled: {len(sampled)} brand entries across {sampled["_generic"].nunique()} generics')

Top 10 generics: ['cefixime', 'domperidone', 'ofloxacin', 'ceftriaxone', 'levocetirizine', 'glimepiride', 'cefuroxime', 'pantoprazole', 'ondansetron', 'rabeprazole']

Sampled: 600 brand entries across 200 generics


## Cell 9 — Build the unified JSON

In [ ]:
records = []

# Indian-brand entries
for _, row in sampled.iterrows():
    info = generic_lookup[row['_generic']]
    records.append({
        'name': str(row['name']),
        'generic': info['generic_name'],
        'query': str(row['name']).lower(),
        'description': info['description'],
        'uses': info['uses'],
        'side_effects': info['side_effects'],
        'how_to_use': '',
        'warnings': '',
        'composition': str(row.get('short_composition1', '')),
        'manufacturer': str(row.get('manufacturer_name', '')),
        'source': 'darkknight25 + junioralive',
    })

# Pure-generic entries
for key in top_generics:
    info = generic_lookup[key]
    records.append({
        'name': info['generic_name'],
        'generic': info['generic_name'],
        'query': info['generic_name'].lower(),
        'description': info['description'],
        'uses': info['uses'],
        'side_effects': info['side_effects'],
        'how_to_use': '',
        'warnings': '',
        'composition': info['generic_name'],
        'manufacturer': '',
        'source': 'darkknight25',
    })

PATHS.drugs_json.write_text(
    json.dumps(records, indent=2, ensure_ascii=False),
    encoding='utf-8'
)
print(f'Wrote {len(records)} records to:\n  {PATHS.drugs_json}')

Wrote 800 records to:
  /content/drive/MyDrive/prescriptai/data/drugs/indian_drugs.json


## Cell 10 — Inspect

In [ ]:
drugs = json.loads(PATHS.drugs_json.read_text(encoding='utf-8'))

print(f'Total records: {len(drugs)}\n')
print(f'{"Field":15s} {"Filled":12s} {"Avg length"}')
print('-' * 40)
for field in ['name', 'generic', 'uses', 'side_effects', 'description']:
    filled = sum(1 for d in drugs if d.get(field))
    avg = sum(len(str(d.get(field, ''))) for d in drugs) / len(drugs)
    print(f'{field:15s} {filled}/{len(drugs):<10} {avg:.0f}')

print('\n--- Sample brand entry ---')
brand_example = next(d for d in drugs if d['name'] != d['generic'])
for k, v in brand_example.items():
    print(f'{k}: {str(v)[:200]}')

print('\n--- Sample generic entry ---')
generic_example = next(d for d in drugs if d['name'] == d['generic'])
for k, v in generic_example.items():
    print(f'{k}: {str(v)[:200]}')

Total records: 800

Field           Filled       Avg length
----------------------------------------
name            800/800        18
generic         800/800        11
uses            800/800        46
side_effects    800/800        38
description     800/800        78

--- Sample brand entry ---
name: Allegra 120mg Tablet
generic: Fexofenadine
query: allegra 120mg tablet
description: Fexofenadine is a second-generation antihistamine used in seasonal allergies.
uses: Allergic rhinitis, Chronic urticaria, Hay fever
side_effects: Drowsiness, Headache, Dry mouth
how_to_use: 
warnings: 
composition: Fexofenadine (120mg)
manufacturer: Sanofi India  Ltd
source: darkknight25 + junioralive

--- Sample generic entry ---
name: Cefixime
generic: Cefixime
query: cefixime
description: Cefixime is a third-generation cephalosporin antibiotic used to treat various infections.
uses: Urinary tract infections, Throat infections, Ear infections
side_effects: Diarrhea, Allergic reactions, Abdominal pain
h

## Module 1 deliverables

- [ ] `data/drugs/indian_drugs.json` exists
- [ ] 600+ records
- [ ] `uses` and `side_effects` filled for ≥95% of records
- [ ] Both Indian brand entries and generic entries present

**Story for the report:** *"The drug corpus was built by joining two open datasets — a curated medical reference set sourced from WHO/MedlinePlus/FDA (700 drugs with uses, side effects) with an Indian commercial drug dataset (~250k brand entries). Records were joined on normalized generic ingredient names, retaining the top 200 most-common generics and up to 3 brand variants per generic."*

When all 4 deliverables are checked, ping me — Module 2 (RAG layer with CLIP + ChromaDB) is next, and that's where your novel research contribution starts.